# Сверхразрешение изображений

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Сверхразрешение изображений».

## Научная цель

Сверхразрешение восстанавливает изображение высокого разрешения из снимка низкого качества. Модель обучают на парах «низкое разрешение — высокое разрешение», и она учится правдоподобно дорисовывать детали. Для учёного это способ извлечь больше информации из имеющихся снимков, ускорить съёмку или снизить дозу облучения образца.

В этом блокноте мы обучим компактную свёрточную сеть сверхразрешения (идея SRCNN) на синтетических изображениях и сравним её с простой бикубической интерполяцией. Блокнот исполняется на CPU за минуту.

**Важное предупреждение.** Восстановленные детали — правдоподобная реконструкция, а не измерение. Их нельзя трактовать как факт без независимой проверки на снимках высокого разрешения.

## Интуиция за методом

Представьте реставратора, который восстанавливает выцветшую фотографию: он смотрит на размытые очертания и, зная, как обычно выглядят подобные объекты, «дорисовывает» утраченные детали. Сверхразрешение — это именно такой процесс, только выполняемый нейронной сетью.

Сеть обучают на тысячах пар «размытый снимок — чёткий оригинал»: она видит размытое, видит ожидаемый резкий результат и учится, какие детали нужно добавить. После обучения она способна «навести резкость» на новый размытый снимок.

**Важное предостережение:** восстановленные детали — это правдоподобная реконструкция на основе статистики обучающих данных, а не физическое измерение. Их нельзя использовать для количественных выводов без независимой проверки.

**Ключевые понятия, которые встретятся в блокноте:**
- **Разрешение (resolution)** — количество пикселей на единицу длины. Высокое разрешение = больше деталей на той же площади.
- **Бикубическая интерполяция** — математический метод увеличения изображения путём сглаживания. Даёт размытый, но без артефактов результат; служит базовым сравнением.
- **PSNR (Peak Signal-to-Noise Ratio, пиковое отношение сигнал/шум)** — измеряется в децибелах (дБ); чем выше, тем ближе восстановленный снимок к оригиналу. Разница в 2+ дБ уже заметна визуально.
- **Остаточное обучение (residual learning)** — сеть предсказывает не само изображение, а «поправку» к размытому входу; это ускоряет обучение.
- **Галлюцинации** — детали, которые сеть добавила, но которых в оригинале нет. Главный риск сверхразрешения.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем синтетические снимки с чёткими структурами — наложение плавных волн и резких границ (характерно для микроскопии и спутниковых снимков). Из каждого изображения высокого разрешения создадим вход низкого разрешения: уменьшим в 2 раза и вернём к исходному размеру — так возникает характерная размытость. Пара «размытый вход — резкий оригинал» — это один обучающий пример.

**Что делает ячейка ниже:** создаёт 200 таких пар и показывает три примера рядом (размытый вход / оригинал) до обучения сети — чтобы было видно, с какой деградацией нужно справиться.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(0)
torch.manual_seed(0)
SIZE = 48          # сторона изображения высокого разрешения
SCALE = 2          # коэффициент уменьшения


def make_hr(_):
    """Синтетическое изображение высокого разрешения."""
    yy, xx = np.mgrid[0:SIZE, 0:SIZE] / SIZE
    fx, fy = rng.uniform(3, 8, size=2)
    ph = rng.uniform(0, 6.28)
    img = 0.5 + 0.3 * np.sin(2 * np.pi * (fx * xx + fy * yy) + ph)
    # Добавим резкую границу — деталь, которую трудно восстановить.
    edge = rng.uniform(0.3, 0.7)
    img[xx > edge] += 0.25
    return np.clip(img, 0, 1).astype('float32')


def degrade(hr):
    """Имитация съёмки низкого разрешения: уменьшение и возврат."""
    t = torch.from_numpy(hr)[None, None]
    small = F.interpolate(t, scale_factor=1 / SCALE, mode='bilinear',
                          align_corners=False, recompute_scale_factor=False)
    lr = F.interpolate(small, size=(SIZE, SIZE), mode='bicubic',
                       align_corners=False)
    return lr[0, 0].clamp(0, 1).numpy().astype('float32')


hr_imgs = np.stack([make_hr(i) for i in range(200)])
lr_imgs = np.stack([degrade(h) for h in hr_imgs])
print(f'Пар изображений: {len(hr_imgs)}, размер: {SIZE}x{SIZE}')

In [ ]:
import matplotlib.pyplot as plt

# Предпросмотр данных: размытый вход vs оригинал.
n_prev = 3
fig, axes = plt.subplots(2, n_prev, figsize=(3.5 * n_prev, 7.0),
                         gridspec_kw={"hspace": 0.06, "wspace": 0.06})
for j in range(n_prev):
    axes[0, j].imshow(lr_imgs[j], cmap="gray", vmin=0, vmax=1)
    axes[1, j].imshow(hr_imgs[j], cmap="gray", vmin=0, vmax=1)
    for i in range(2):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        axes[i, j].grid(False)
axes[0, 0].set_ylabel("Вход (низкое\nразрешение)", fontsize=11)
axes[1, 0].set_ylabel("Оригинал (высокое\nразрешение)", fontsize=11)
fig.suptitle("Обучающие пары: что даём на вход и что хотим получить",
             fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график.** Верхний ряд — размытые входы (бикубическое уменьшение и обратное увеличение): границы волн нечёткие, детали смазаны. Нижний ряд — оригиналы высокого разрешения: резкие границы, чёткий переход яркости. Задача сети — превратить верхний ряд в нижний.

## 4. Применение метода

### Шаг 1. Архитектура сети (SRCNN)

**Что делает ячейка ниже:** определяет компактную свёрточную сеть сверхразрешения. Идея SRCNN (Super-Resolution Convolutional Neural Network): три свёрточных слоя последовательно извлекают признаки из размытого входа, а итоговый слой собирает из них уточнённое изображение.

В нашей реализации сеть предсказывает не само изображение, а **поправку** к размытому входу (остаточное обучение): выход = вход + поправка. Это упрощает обучение, потому что большинство пикселей нужно изменить незначительно.

In [ ]:
import torch.nn as nn


class SRCNN(nn.Module):
    """Компактная свёрточная сеть сверхразрешения."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 5, padding=2), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, x):
        # Сеть предсказывает поправку к размытому входу (остаток).
        return (x + self.net(x)).clamp(0, 1)


model = SRCNN()
print(model)

### Шаг 2. Обучение

**Что делает ячейка ниже:** обучает сеть на 160 парах изображений в течение 40 эпох. Функция потерь — MSE (среднеквадратическая ошибка по пикселям): модель минимизирует разницу между своим предсказанием и оригиналом. Уменьшение потерь означает, что восстановление становится точнее.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

n_train = 160
ds = TensorDataset(
    torch.from_numpy(lr_imgs[:n_train])[:, None],
    torch.from_numpy(hr_imgs[:n_train])[:, None])
loader = DataLoader(ds, batch_size=16, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.MSELoss()

history = []
for epoch in range(1, 41):
    model.train()
    ep = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        ep += loss.item() * len(xb)
    history.append(ep / n_train)
    if epoch % 10 == 0:
        print(f'Эпоха {epoch:2d}: потеря восстановления {history[-1]:.5f}')

### Шаг 3. Оценка качества и визуализация «до / после / оригинал»

**Что делает ячейка ниже:** вычисляет PSNR для бикубической интерполяции и для нашей сети на всех 40 тестовых изображениях. Затем показывает три строки рядом: размытый вход, восстановленный сетью и оригинал — с подписями PSNR под каждым.

**PSNR (в дБ):** чем выше, тем лучше. Бикубическая интерполяция задаёт базовую линию — «насколько хорошо и без нейросети». Сеть должна её превысить.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def psnr(a, b):
    """Пиковое отношение сигнал/шум для изображений в [0, 1]."""
    mse = np.mean((a - b) ** 2)
    return 99.0 if mse < 1e-12 else 10 * np.log10(1.0 / mse)


model.eval()
test_lr = lr_imgs[n_train:]
test_hr = hr_imgs[n_train:]
import torch
with torch.no_grad():
    sr = model(torch.from_numpy(test_lr)[:, None])[:, 0].numpy()

psnr_lr_all = [psnr(l, h) for l, h in zip(test_lr, test_hr)]
psnr_sr_all = [psnr(s, h) for s, h in zip(sr, test_hr)]
psnr_lr_mean = np.mean(psnr_lr_all)
psnr_sr_mean = np.mean(psnr_sr_all)

print(f"PSNR бикубической интерполяции (вход):  {psnr_lr_mean:.2f} дБ")
print(f"PSNR сети сверхразрешения (восстановл.): {psnr_sr_mean:.2f} дБ")
print(f"Улучшение: +{psnr_sr_mean - psnr_lr_mean:.2f} дБ")

# Визуализация «до / после / оригинал» для трёх тестовых примеров.
n_show = 3
row_labels = ["Вход (низкое разрешение)", "Восстановлено сетью",
              "Оригинал (высокое разрешение)"]
fig, axes = plt.subplots(3, n_show, figsize=(3.8 * n_show, 11.0),
                         gridspec_kw={"hspace": 0.08, "wspace": 0.06})
for j in range(n_show):
    lr_img = test_lr[j]
    sr_img = sr[j]
    hr_img = test_hr[j]
    psnr_in = psnr_lr_all[j]
    psnr_out = psnr_sr_all[j]

    axes[0, j].imshow(lr_img, cmap="gray", vmin=0, vmax=1)
    axes[0, j].set_xlabel(f"PSNR = {psnr_in:.1f} дБ", fontsize=11,
                          color=VIZ["series"][3])
    axes[1, j].imshow(sr_img, cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_xlabel(f"PSNR = {psnr_out:.1f} дБ  (+{psnr_out - psnr_in:.1f} дБ)",
                          fontsize=11, color=VIZ["series"][1], fontweight="bold")
    axes[2, j].imshow(hr_img, cmap="gray", vmin=0, vmax=1)
    axes[2, j].set_xlabel("Эталон", fontsize=11, color=VIZ["series"][0])
    for i in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        axes[i, j].grid(False)

for i, label in enumerate(row_labels):
    axes[i, 0].set_ylabel(label, fontsize=10, labelpad=6)

fig.suptitle(
    f"Сверхразрешение: вход vs. восстановление vs. оригинал\n"
    f"Средний PSNR: вход {psnr_lr_mean:.1f} дБ  →  сеть {psnr_sr_mean:.1f} дБ",
    fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

**Как читать этот график.**

- **Верхний ряд (вход):** размытые изображения с низким PSNR — именно то, с чем работает сеть. Серое число дБ показывает «расстояние» до оригинала.
- **Средний ряд (восстановление сетью):** резкость должна заметно вырасти; зелёное число дБ и прирост в скобках показывают, насколько сеть приблизилась к оригиналу по сравнению с входом. Прирост +1 дБ уже заметен глазу.
- **Нижний ряд (оригинал):** эталон. Чем ближе средний ряд к нижнему — тем лучше сработала сеть.

Обращайте внимание не только на PSNR, но и на резкость границ — это именно то, что восстанавливает сверхразрешение.

## 5. Интерпретация результата

- **PSNR сети выше**, чем у интерполяции: модель восстанавливает резкость границ и детали лучше простого сглаживания.
- **Три изображения рядом** позволяют качественно оценить результат: восстановленный кадр должен быть ближе к оригиналу, чем размытый вход.
- Метрики PSNR/SSIM измеряют близость к эталону, но **не гарантируют научную достоверность** отдельных восстановленных деталей.
- Главный риск — **галлюцинации**: модель может убедительно дорисовать структуры, которых в образце нет. Для количественного анализа результат сверхразрешения нельзя считать измерением.

Корректное применение: сверхразрешение — средство визуализации и предобработки, а научные выводы проверяются независимо на снимках высокого разрешения.

## 4а. Поэкспериментируйте сами

| Что изменить | Где | Ожидаемый эффект |
|---|---|---|
| `SCALE = 4` вместо 2 | ячейка данных | Более сильная деградация; сложнее — PSNR ниже |
| Добавить слой в `SRCNN` (ещё один `Conv2d(32,32,3)`) | ячейка модели | Больше параметров — возможно лучше, но дольше |
| `n_train = 40` | ячейка обучения | Мало данных — переобучение; смотрите на тестовый PSNR |
| В `make_hr`: убрать строку с `edge` (без резкой границы) | ячейка данных | Задача легче — сеть покажет более высокий PSNR |

## 6. Попробуйте на своих данных

1. Подготовьте пары изображений: вход низкого разрешения и эталон высокого. Критично, чтобы модель деградации в парах **совпадала с реальной** деградацией ваших снимков.
2. Приведите изображения к одному размеру и нормируйте яркость в диапазон [0, 1].
3. Снимите комментарии в ячейке ниже и подставьте свои массивы.
4. Обучение только на синтетически уменьшенных изображениях плохо переносится на реальные данные — по возможности используйте реальные пары съёмки.

In [ ]:
# --- Шаблон загрузки своих пар изображений ---
# from pathlib import Path
# from PIL import Image
#
# def load_gray(path, size):
#     img = Image.open(path).convert('L').resize((size, size))
#     return np.asarray(img, dtype='float32') / 255.0
#
# hr_imgs = np.stack([load_gray(p, SIZE)
#                     for p in Path('hr').glob('*')])
# lr_imgs = np.stack([load_gray(p, SIZE)
#                     for p in Path('lr').glob('*')])
# Далее повторите обучение из раздела 4.

## 7. Краткие выводы

- Сверхразрешение — это восстановление деталей из размытого/малоразмерного снимка с помощью нейронной сети, обученной на парах «плохой — хороший снимок».
- PSNR измеряет близость к оригиналу в децибелах; сеть должна превосходить бикубическую интерполяцию как минимум на 1–2 дБ, чтобы улучшение имело смысл.
- Восстановленные детали — правдоподобная реконструкция, а не измерение. Главный риск — галлюцинации: сеть может убедительно дорисовать структуры, которых нет.
- Для научного применения результат сверхразрешения нужно валидировать на независимых снимках высокого разрешения того же объекта.
- Качество сильно зависит от того, насколько деградация в обучающих парах совпадает с реальной деградацией ваших снимков.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Сеть дала прирост PSNR +2.5 дБ по сравнению с бикубической интерполяцией, однако визуально восстановленное изображение выглядит мыльным, а некоторые «восстановленные» детали отсутствуют в оригинале. Как соотнести это с показателем PSNR и что это говорит о применимости сети к научным измерениям?
2. Сеть обучена на синтетических парах, полученных бикубическим уменьшением, но применяется к реальным микроскопическим снимкам, деградация которых обусловлена дифракционным пределом и шумом детектора. Какой конкретный эффект следует ожидать и как проверить, пригоден ли суррогат для этой задачи?
3. Вы удвоили число обучающих пар со 160 до 320 и добавили второй свёрточный блок в SRCNN. PSNR на тестовой выборке вырос на 0.3 дБ, но время инференса увеличилось в три раза. Какой критерий позволяет решить, оправдан ли этот компромисс для вашей конкретной научной задачи?

<details>
<summary>Показать ориентиры для ответов</summary>

1. PSNR — среднеквадратическая метрика по всем пикселям: она растёт при уменьшении средней ошибки, но не штрафует за «правдоподобные галлюцинации», которые визуально убедительны, однако физически отсутствуют. Для научных измерений это принципиально: восстановленные детали нельзя трактовать как реальные структуры без независимой проверки на снимке высокого разрешения.
2. Сеть, обученная на бикубической деградации, не знает, как выглядит дифракционная размытость или дробный шум детектора. На реальных микроскопических снимках она будет вводить артефакты или не улучшать изображение. Проверка: сформировать несколько реальных пар «до/после» (например, снять объект при полной и сниженной экспозиции) и измерить PSNR/SSIM на них, а не на синтетических парах.
3. Ключевой критерий — не абсолютный PSNR, а практическая значимость: можно ли по дополнительным 0.3 дБ достоверно разрешить структуры, критичные для вывода (например, различить две соседние органеллы)? Если нет, прирост не оправдывает замедление. Дополнительный аргумент: для потокового анализа данных (высокоскоростная микроскопия) время инференса может быть жёстким ограничением.
</details>